In [ ]:
!pip install datasets transformers evaluate sentencepiece accelerate gradio rouge_score sacrebleu scikit-learn -q

from datasets import load_dataset
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
raw_dataset = load_dataset("truehealth/medicationqa")

split_dataset = raw_dataset["train"].train_test_split(test_size=0.2, seed=42)
temp_dataset = split_dataset["test"].train_test_split(test_size=0.5, seed=42)

dataset = {
    "train": split_dataset["train"],
    "validation": temp_dataset["train"],
    "test": temp_dataset["test"]
}

len(dataset["train"]), len(dataset["validation"]), len(dataset["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001-7427a10e891759(…):   0%|          | 0.00/215k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/690 [00:00<?, ? examples/s]

(552, 69, 69)

In [ ]:
def clean_dataset(example):
    example["Question"] = str(example["Question"]).strip()
    example["Answer"] = str(example["Answer"]).strip()
    return example

dataset["train"] = dataset["train"].map(clean_dataset)
dataset["validation"] = dataset["validation"].map(clean_dataset)
dataset["test"] = dataset["test"].map(clean_dataset)

dataset["train"] = dataset["train"].filter(lambda x: x["Question"] != "" and x["Answer"] != "")
dataset["validation"] = dataset["validation"].filter(lambda x: x["Question"] != "" and x["Answer"] != "")
dataset["test"] = dataset["test"].filter(lambda x: x["Question"] != "" and x["Answer"] != "")

len(dataset["train"]), len(dataset["validation"]), len(dataset["test"])

Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Filter:   0%|          | 0/552 [00:00<?, ? examples/s]

Filter:   0%|          | 0/69 [00:00<?, ? examples/s]

Filter:   0%|          | 0/69 [00:00<?, ? examples/s]

(552, 69, 69)

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model_t5 = T5ForConditionalGeneration.from_pretrained("t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_t5(examples):
    inputs = ["question: " + str(q) for q in examples["Question"]]
    targets = [str(a) for a in examples["Answer"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_t5 = dataset["train"].map(preprocess_t5, batched=True)
tokenized_val_t5 = dataset["validation"].map(preprocess_t5, batched=True)
tokenized_test_t5 = dataset["test"].map(preprocess_t5, batched=True)

Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator_t5 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_t5)

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir="./t5_medicationqa_model",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none"
)

trainer_t5 = Seq2SeqTrainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=tokenized_train_t5,
    eval_dataset=tokenized_val_t5,
    processing_class=tokenizer,
    data_collator=data_collator_t5
)

trainer_t5.train()

Epoch,Training Loss,Validation Loss
1,2.341277,1.998230
2,2.026210,1.908534
3,2.286203,1.892051


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=414, training_loss=2.3100019998596486, metrics={'train_runtime': 58.1292, 'train_samples_per_second': 28.488, 'train_steps_per_second': 7.122, 'total_flos': 56031505809408.0, 'train_loss': 2.3100019998596486, 'epoch': 3.0})

In [ ]:
trainer_t5.save_model("/content/drive/MyDrive/t5_medicationqa_model")
tokenizer.save_pretrained("/content/drive/MyDrive/t5_medicationqa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/t5_medicationqa_model/tokenizer_config.json',
 '/content/drive/MyDrive/t5_medicationqa_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_flan = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_flan = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess_flan(examples):
    inputs = ["question: " + str(q) for q in examples["Question"]]
    targets = [str(a) for a in examples["Answer"]]

    model_inputs = tokenizer_flan(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer_flan(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_flan = dataset["train"].map(preprocess_flan, batched=True)
tokenized_val_flan = dataset["validation"].map(preprocess_flan, batched=True)
tokenized_test_flan = dataset["test"].map(preprocess_flan, batched=True)

Map:   0%|          | 0/552 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

In [ ]:
data_collator_flan = DataCollatorForSeq2Seq(tokenizer=tokenizer_flan, model=model_flan)

training_args_flan = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_medicationqa_model",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none"
)

trainer_flan = Seq2SeqTrainer(
    model=model_flan,
    args=training_args_flan,
    train_dataset=tokenized_train_flan,
    eval_dataset=tokenized_val_flan,
    processing_class=tokenizer_flan,
    data_collator=data_collator_flan
)

trainer_flan.train()

Epoch,Training Loss,Validation Loss
1,7.541355,6.800248
2,6.554834,6.195823
3,6.609808,6.043949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=414, training_loss=7.249480519317775, metrics={'train_runtime': 81.4037, 'train_samples_per_second': 20.343, 'train_steps_per_second': 5.086, 'total_flos': 76958668947456.0, 'train_loss': 7.249480519317775, 'epoch': 3.0})

In [ ]:
trainer_flan.save_model("/content/drive/MyDrive/flan_t5_medicationqa_model")
tokenizer_flan.save_pretrained("/content/drive/MyDrive/flan_t5_medicationqa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/flan_t5_medicationqa_model/tokenizer_config.json',
 '/content/drive/MyDrive/flan_t5_medicationqa_model/tokenizer.json')

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

In [ ]:
def generate_answers(model, tokenizer, questions):
    inputs = ["question: " + str(q) for q in questions]

    encoded = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    outputs = model.generate(
        **encoded,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return predictions

test_questions = dataset["test"]["Question"]
test_answers = [str(a) for a in dataset["test"]["Answer"]]

t5_predictions = generate_answers(model_t5, tokenizer, test_questions)
flan_predictions = generate_answers(model_flan, tokenizer_flan, test_questions)

In [ ]:
t5_rouge = rouge.compute(predictions=t5_predictions, references=test_answers)
flan_rouge = rouge.compute(predictions=flan_predictions, references=test_answers)

t5_bleu = bleu.compute(predictions=t5_predictions, references=[[a] for a in test_answers])
flan_bleu = bleu.compute(predictions=flan_predictions, references=[[a] for a in test_answers])

results = {
    "Model": ["T5-small", "FLAN-T5-small"],
    "ROUGE-1": [t5_rouge["rouge1"], flan_rouge["rouge1"]],
    "ROUGE-2": [t5_rouge["rouge2"], flan_rouge["rouge2"]],
    "ROUGE-L": [t5_rouge["rougeL"], flan_rouge["rougeL"]],
    "BLEU": [t5_bleu["bleu"], flan_bleu["bleu"]]
}

results_df = pd.DataFrame(results)
results_df

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BLEU
0,T5-small,0.106462,0.015700,0.088951,0.0
1,FLAN-T5-small,0.016862,0.000225,0.015682,0.0


In [ ]:
best_model = results_df.sort_values(by="ROUGE-L", ascending=False).iloc[0]
best_model

,0
Model,T5-small
ROUGE-1,0.106462
ROUGE-2,0.0157
ROUGE-L,0.088951
BLEU,0.0


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

questions = [str(q) for q in dataset["train"]["Question"]]
answers = [str(a) for a in dataset["train"]["Answer"]]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(questions)

def tfidf_chatbot(user_question):
    user_vec = vectorizer.transform([user_question])
    similarity = cosine_similarity(user_vec, tfidf_matrix)
    index = similarity.argmax()
    return answers[index]

In [ ]:
import gradio as gr

def chatbot(question, model_choice):
    if model_choice == "T5-small":
        model = model_t5
        tok = tokenizer
    elif model_choice == "FLAN-T5-small":
        model = model_flan
        tok = tokenizer_flan
    else:
        return tfidf_chatbot(question)

    inputs = tok(
        "question: " + question,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4
    )

    answer = tok.decode(outputs[0], skip_special_tokens=True)
    return answer

interface = gr.Interface(
    fn=chatbot,
    inputs=[
        gr.Textbox(label="Soru", lines=3),
        gr.Radio(["T5-small", "FLAN-T5-small", "TF-IDF"], label="Model Seç")
    ],
    outputs=gr.Textbox(label="Cevap", lines=8, max_lines=12),
    title="MedicationQA Chatbot"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c33fc9a5ee6ea62013.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
